# PlacementIQ — Notebook 01: Data Audit

Objective: Validate the raw dataset schema, dimensions, missing values, duplicates, data types, and target distribution.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ──────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/raw/student_placement_synthetic.csv')
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'\nColumn dtypes:')
print(df.dtypes)
print(f'\nFirst 5 rows:')
df.head(5)

## 2. Schema Validation

In [ ]:
# Expected schema
REQUIRED_COLUMNS = [
    'branch', 'college_tier', 'cgpa', 'backlogs', 'coding_skills',
    'dsa_score', 'aptitude_score', 'communication_skills', 'ml_knowledge',
    'system_design', 'internships', 'projects_count', 'certifications',
    'hackathons', 'open_source_contributions', 'extracurriculars',
    'placement_status', 'salary_package_lpa'
]

EXPECTED_DTYPES = {
    'branch': 'object',
    'college_tier': 'object',
    'cgpa': 'float64',
    'backlogs': 'int64',
    'coding_skills': 'float64',
    'dsa_score': 'float64',
    'aptitude_score': 'float64',
    'communication_skills': 'float64',
    'ml_knowledge': 'float64',
    'system_design': 'float64',
    'internships': 'int64',
    'projects_count': 'int64',
    'certifications': 'int64',
    'hackathons': 'int64',
    'open_source_contributions': 'int64',
    'extracurriculars': 'int64',
    'placement_status': 'int64',
}

# Check required columns present
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
print('=== Column Presence Check ===')
if missing_cols:
    print(f'  FAIL — Missing columns: {missing_cols}')
else:
    print(f'  PASS — All {len(REQUIRED_COLUMNS)} required columns present')

# Verify dtypes
print('\n=== Dtype Validation ===')
dtype_issues = []
for col, expected in EXPECTED_DTYPES.items():
    actual = str(df[col].dtype)
    # Allow int32/int64 interchangeably
    match = (actual == expected) or (expected == 'int64' and actual in ('int32', 'int64'))
    if not match:
        dtype_issues.append(f'  {col}: expected {expected}, got {actual}')
if dtype_issues:
    print('  WARN — Dtype mismatches:')
    for issue in dtype_issues:
        print(issue)
else:
    print('  PASS — All dtypes match expected schema')

# Check salary_package_lpa NaN count
salary_nan_count = df['salary_package_lpa'].isna().sum()
print(f'\n=== salary_package_lpa NaN Check ===')
print(f'  NaN count : {salary_nan_count}')
print(f'  Expected  : 31525 (one per unplaced student)')
if salary_nan_count == 31525:
    print('  PASS — NaN count matches unplaced count')
else:
    print('  WARN — NaN count does not match expected 31525')

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('Columns with missing values:')
if missing_df.empty:
    print('  None (excluding salary_package_lpa)')
else:
    display(missing_df.style.background_gradient(cmap='Reds'))

print('\nNote: salary_package_lpa is EXCLUDED from modelling — it is a post-placement'
      ' leakage column (NaN for all unplaced students).')

## 4. Duplicate Detection

In [ ]:
dup_count = df.duplicated().sum()
print(f'Total duplicate rows: {dup_count}')
if dup_count == 0:
    print('PASS — No duplicate rows detected.')
else:
    print(f'WARN — {dup_count} duplicate rows found. Review before modelling.')

## 5. Target Variable Distribution

In [ ]:
import matplotlib.pyplot as plt

target_counts = df['placement_status'].value_counts()
target_pct = df['placement_status'].value_counts(normalize=True) * 100

print('=== Target Variable: placement_status ===')
for label, count in target_counts.items():
    status_name = 'Placed' if label == 1 else 'Not Placed'
    print(f'  {status_name} ({label}): {count:,}  ({target_pct[label]:.2f}%)')

# Side-by-side pie + bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ['Placed (1)', 'Not Placed (0)']
sizes = [target_counts.get(1, 0), target_counts.get(0, 0)]
colors = ['#4CAF50', '#F44336']

# Pie chart
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.2f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Placement Status — Pie Chart', fontsize=13, fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.2)
for bar, count in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
                 f'{count:,}', ha='center', va='bottom', fontsize=11)
axes[1].set_title('Placement Status — Bar Chart', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_ylim(0, max(sizes) * 1.15)

plt.suptitle('Target Variable Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to reports/figures/01_target_distribution.png')

## 6. Categorical Features

In [ ]:
print('=== Branch ===')
print(df['branch'].value_counts())
print(f'Unique values: {sorted(df["branch"].unique().tolist())}')

print('\n=== College Tier ===')
print(df['college_tier'].value_counts())
print(f'Unique values: {sorted(df["college_tier"].unique().tolist())}')

## 7. Numerical Feature Statistics

In [ ]:
NUMERICAL_FEATURES = [
    'cgpa', 'backlogs', 'coding_skills', 'dsa_score', 'aptitude_score',
    'communication_skills', 'ml_knowledge', 'system_design', 'internships',
    'projects_count', 'certifications', 'hackathons',
    'open_source_contributions', 'extracurriculars'
]

stats = df[NUMERICAL_FEATURES].describe().T.round(2)
stats['dtype'] = df[NUMERICAL_FEATURES].dtypes.astype(str)
display(stats)

## 8. Range Validation

In [ ]:
# Documented bounds per feature
BOUNDS = {
    'cgpa':                      (4.0, 10.0),
    'backlogs':                  (0,   3),
    'coding_skills':             (1.0, 10.0),
    'dsa_score':                 (1.0, 10.0),
    'aptitude_score':            (20.0, 100.0),
    'communication_skills':      (1.0, 10.0),
    'ml_knowledge':              (0.0, 10.0),
    'system_design':             (0.0, 10.0),
    'internships':               (0,   3),
    'projects_count':            (0,   5),
    'certifications':            (0,   4),
    'hackathons':                (0,   3),
    'open_source_contributions': (0,   2),
    'extracurriculars':          (0,   3),
}

print('=== Range Validation ===')
issues_found = False
for feature, (lo, hi) in BOUNDS.items():
    out_of_range = df[(df[feature] < lo) | (df[feature] > hi)]
    if len(out_of_range) > 0:
        print(f'  FAIL  {feature:30s}: {len(out_of_range)} values outside [{lo}, {hi}]')
        issues_found = True
    else:
        print(f'  PASS  {feature:30s}: all values within [{lo}, {hi}]')

if not issues_found:
    print('\nAll numerical features are within documented bounds.')

## 9. Audit Summary

In [ ]:
import json

placed_count = int(df['placement_status'].sum())
not_placed_count = int((df['placement_status'] == 0).sum())
total = len(df)

audit_summary = {
    'shape': {'rows': total, 'columns': df.shape[1]},
    'missing_values': {
        'salary_package_lpa': int(df['salary_package_lpa'].isna().sum()),
        'other_columns': int(df.drop(columns=['salary_package_lpa']).isnull().sum().sum())
    },
    'duplicates': int(df.duplicated().sum()),
    'target_distribution': {
        'placed_1': placed_count,
        'not_placed_0': not_placed_count,
        'placed_pct': round(placed_count / total * 100, 2),
        'not_placed_pct': round(not_placed_count / total * 100, 2)
    },
    'schema_issues': len(missing_cols) + len(dtype_issues),
    'range_violations': 0,  # Update if issues_found above
    'verdict': 'PASS — Dataset is clean and ready for preprocessing.'
}

print('=== AUDIT SUMMARY ===')
print(json.dumps(audit_summary, indent=2))